In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# ------------------------------------------------------------
# PROJECT PATHS
# ------------------------------------------------------------
PROJECT_ROOT = Path("/home/jetson/Desktop/ciesol-anomaly-detection")

MODEL1_PATH = PROJECT_ROOT / "models" / "lstm_classifier_best.keras"
MODEL2_PATH = PROJECT_ROOT / "models" / "model2_v2" / "lstm_model2_best.keras"

SEQUENCE_DIR = PROJECT_ROOT / "data" / "sequences"
MODEL2_SEQUENCE_DIR = SEQUENCE_DIR / "model2_v2"

MODEL2_DATA_DIR = PROJECT_ROOT / "data" / "model2"
MODEL2_SCALED_DIR = PROJECT_ROOT / "data" / "scaled" / "model2"

# Model 1 binary decision threshold from your evaluation
MODEL1_THRESHOLD = 0.24

SEQUENCE_LENGTH = 20

CLASS_NAMES = [
    "Contextual",
    "Ruido",
    "Desviacion de Correlacion",
    "Datos Faltantes",
    "Valores Fuera de Rango",
    "Sensor Atascado",
]

print("TensorFlow:", tf.__version__)
print("Project root exists:", PROJECT_ROOT.exists())
print("TensorFlow GPU devices:", tf.config.list_physical_devices("GPU"))

assert MODEL1_PATH.exists(), f"Model 1 not found: {MODEL1_PATH}"
assert MODEL2_PATH.exists(), f"Model 2 not found: {MODEL2_PATH}"

In [ ]:
print("MODEL 1 SEQUENCES")
for path in sorted(SEQUENCE_DIR.glob("*test*")):
    print(" ", path.relative_to(PROJECT_ROOT))

print("\nMODEL 2 SEQUENCES")
for path in sorted(MODEL2_SEQUENCE_DIR.glob("*test*")):
    print(" ", path.relative_to(PROJECT_ROOT))

print("\nMODEL 2 PREPROCESSING ARTIFACTS")
if MODEL2_SCALED_DIR.exists():
    for path in sorted(MODEL2_SCALED_DIR.iterdir()):
        print(" ", path.name)
else:
    print("Directory not found:", MODEL2_SCALED_DIR)

print("\nMODEL 2 TEST TABLE")
print((MODEL2_DATA_DIR / "test_model2.parquet").exists())

In [ ]:
model1 = tf.keras.models.load_model(MODEL1_PATH)
model2 = tf.keras.models.load_model(MODEL2_PATH)

print("Model 1 loaded")
model1.summary()

print("\nModel 2 loaded")
model2.summary()

In [ ]:
# Model 1: all test sequences
X1_test = np.load(SEQUENCE_DIR / "test_lstm_sequences_X.npy")
y1_test = np.load(SEQUENCE_DIR / "test_lstm_sequences_y.npy")

# Model 2 v2: saved multiclass test sequences
X2_test = np.load(MODEL2_SEQUENCE_DIR / "test_model2_X.npy")
y2_test = np.load(MODEL2_SEQUENCE_DIR / "test_model2_y.npy")

X1_test = np.asarray(X1_test, dtype=np.float32)
X2_test = np.asarray(X2_test, dtype=np.float32)

y1_test = np.asarray(y1_test).reshape(-1)
y2_test = np.asarray(y2_test).reshape(-1)

print("Model 1 X:", X1_test.shape, X1_test.dtype)
print("Model 1 y:", y1_test.shape, np.unique(y1_test, return_counts=True))

print("\nModel 2 X:", X2_test.shape, X2_test.dtype)
print("Model 2 y:", y2_test.shape, np.unique(y2_test, return_counts=True))

assert X1_test.ndim == 3
assert X2_test.ndim == 3
assert X1_test.shape[1] == SEQUENCE_LENGTH
assert X2_test.shape[1] == SEQUENCE_LENGTH
assert len(X1_test) == len(y1_test)
assert len(X2_test) == len(y2_test)

assert X1_test.shape[2] == model1.input_shape[-1], (
    f"Model 1 expects {model1.input_shape[-1]} features, "
    f"but test data has {X1_test.shape[2]}"
)

assert X2_test.shape[2] == model2.input_shape[-1], (
    f"Model 2 expects {model2.input_shape[-1]} features, "
    f"but test data has {X2_test.shape[2]}"
)

In [ ]:
test_table_path = MODEL2_DATA_DIR / "test_model2.parquet"

test_df = pd.read_parquet(test_table_path)

print("Complete Model 2 test table:", test_df.shape)
print("First columns:", test_df.columns[:25].tolist())

print("\nCandidate scaler/feature files:")
for pattern in ("*.joblib", "*.pkl", "*.pickle", "*.json"):
    for path in sorted(MODEL2_SCALED_DIR.rglob(pattern)):
        print(path.relative_to(PROJECT_ROOT))

In [ ]:
import json
import joblib
import numpy as np

MODEL2_SCALED_DIR = PROJECT_ROOT / "data" / "scaled" / "model2"

SCALER_PATH = MODEL2_SCALED_DIR / "model2_scaler.joblib"
FEATURES_PATH = MODEL2_SCALED_DIR / "model2_features.json"

scaler_model2 = joblib.load(SCALER_PATH)

with open(FEATURES_PATH, "r", encoding="utf-8") as f:
    MODEL2_FINAL_FEATURES = json.load(f)

# Some feature-list JSON files store the list inside a dictionary.
if isinstance(MODEL2_FINAL_FEATURES, dict):
    print("Feature JSON keys:", MODEL2_FINAL_FEATURES.keys())
    possible_keys = ["features", "MODEL2_FINAL_FEATURES", "model2_features"]
    for key in possible_keys:
        if key in MODEL2_FINAL_FEATURES:
            MODEL2_FINAL_FEATURES = MODEL2_FINAL_FEATURES[key]
            break

assert isinstance(MODEL2_FINAL_FEATURES, list), (
    "Could not identify the ordered feature list inside model2_features.json."
)
assert len(MODEL2_FINAL_FEATURES) == model2.input_shape[-1], (
    f"Feature count mismatch: JSON has {len(MODEL2_FINAL_FEATURES)}, "
    f"Model 2 expects {model2.input_shape[-1]}."
)

print("Scaler loaded:", type(scaler_model2).__name__)
print("Model 2 feature count:", len(MODEL2_FINAL_FEATURES))
print("First 10 features:", MODEL2_FINAL_FEATURES[:10])

In [ ]:
# ============================================================
# LOAD SAVED MODEL 2 TEST DATA
# ============================================================

import numpy as np
import pandas as pd

X2_test_rows_scaled = np.load(
    MODEL2_SCALED_DIR / "X_test_model2_scaled.npy",
    mmap_mode="r",
)

# Load the labels saved with the Model 2 scaled data.
# allow_pickle=True is needed because this file contains objects.
y2_scaled_saved = np.load(
    MODEL2_SCALED_DIR / "y_test_model2.npy",
    allow_pickle=True,
)

test_df = pd.read_parquet(
    PROJECT_ROOT / "data" / "model2" / "test_model2.parquet"
)

print("Scaled Model 2 test rows:", X2_test_rows_scaled.shape)
print("Saved labels shape:", y2_scaled_saved.shape)
print("Saved labels dtype:", y2_scaled_saved.dtype)
print("Saved labels sample:", y2_scaled_saved[:5])
print("Model 2 test table:", test_df.shape)
print("Model 1 test sequences:", X1_test.shape)
print("Model 2 expects:", model2.input_shape)

In [ ]:
print("Scaled test rows:", X2_test_rows_scaled.shape)
print("Model 2 sequence length:", model2.input_shape[1])
print("Model 2 feature count:", model2.input_shape[2])
print("Model 1 test sequences:", X1_test.shape)

In [ ]:
# ============================================================
# CREATE ALL MODEL 2 TEST SEQUENCES
# ============================================================

SEQUENCE_LENGTH = model2.input_shape[1]

X2_all_test = np.lib.stride_tricks.sliding_window_view(
    np.asarray(X2_test_rows_scaled, dtype=np.float32),
    window_shape=SEQUENCE_LENGTH,
    axis=0,
)

# sliding_window_view with axis=0 returns:
# (n_sequences, n_features, sequence_length)
# Rearrange to: (n_sequences, sequence_length, n_features)
X2_all_test = np.transpose(X2_all_test, (0, 2, 1)).copy()

print("X2_all_test shape:", X2_all_test.shape)

assert X2_all_test.shape[1:] == tuple(model2.input_shape[1:])
assert len(X2_all_test) == len(X1_test), (
    "Model 1 and Model 2 test sequences are not aligned."
)

In [ ]:
# Separate compiled functions for single-sequence latency tests.
# The batch dimension is fixed at 1 for these functions.

@tf.function(
    input_signature=[
        tf.TensorSpec(
            shape=[1, SEQUENCE_LENGTH, model1.input_shape[-1]],
            dtype=tf.float32,
        )
    ],
    jit_compile=True,
)
def model1_xla_one(x):
    return model1(x, training=False)


@tf.function(
    input_signature=[
        tf.TensorSpec(
            shape=[1, SEQUENCE_LENGTH, model2.input_shape[-1]],
            dtype=tf.float32,
        )
    ],
    jit_compile=True,
)
def model2_xla_one(x):
    return model2(x, training=False)


# Warm-up triggers XLA compilation. Do not include this time in latency.
_ = model1_xla_one(tf.convert_to_tensor(X1_test[:1]))
_ = model2_xla_one(tf.convert_to_tensor(X2_all_test[:1]))

print("XLA functions compiled and warmed up.")

In [ ]:
# Direct TensorFlow calls on tensors; use batches for efficient evaluation.
# This function does not use .predict().

@tf.function(jit_compile=True, reduce_retracing=True)
def model1_xla_batch(x):
    return model1(x, training=False)


@tf.function(jit_compile=True, reduce_retracing=True)
def model2_xla_batch(x):
    return model2(x, training=False)


def predict_in_batches(model_fn, X, batch_size=256):
    outputs = []

    for start in range(0, len(X), batch_size):
        batch = tf.convert_to_tensor(
            X[start : start + batch_size],
            dtype=tf.float32,
        )
        outputs.append(model_fn(batch).numpy())

    return np.concatenate(outputs, axis=0)


# Warm up batch-compiled functions.
_ = model1_xla_batch(tf.convert_to_tensor(X1_test[:min(256, len(X1_test))]))
_ = model2_xla_batch(tf.convert_to_tensor(X2_all_test[:min(256, len(X2_all_test))]))

start = time.perf_counter()
p1 = predict_in_batches(model1_xla_batch, X1_test, batch_size=256)
model1_full_seconds = time.perf_counter() - start

p1 = np.asarray(p1)
if p1.ndim == 2 and p1.shape[1] == 1:
    p1 = p1[:, 0]

# Binary model: probability of anomaly
y1_pred = (p1 >= MODEL1_THRESHOLD).astype(np.int32)

print(f"Model 1 full-test inference: {model1_full_seconds:.3f} s")
print(f"Average per sequence: {model1_full_seconds / len(X1_test) * 1000:.3f} ms")
print(f"Predicted anomalies: {y1_pred.sum()} / {len(y1_pred)}")
print(f"Actual anomalies:    {(y1_test == 1).sum()} / {len(y1_test)}")

print("\nModel 1 classification report:")
print(classification_report(
    y1_test,
    y1_pred,
    labels=[0, 1],
    target_names=["Normal", "Anomaly"],
    digits=4,
    zero_division=0,
))

print("Model 1 confusion matrix:")
print(confusion_matrix(y1_test, y1_pred, labels=[0, 1]))

In [ ]:
# Model 2 receives every sequence flagged by Model 1.
# This includes false positives, as it would in the real pipeline.

model2_indices = np.flatnonzero(y1_pred == 1)

if len(model2_indices) > 0:
    start = time.perf_counter()

    p2_selected = predict_in_batches(
        model2_xla_batch,
        X2_all_test[model2_indices],
        batch_size=256,
    )

    model2_routed_seconds = time.perf_counter() - start
    y2_pred_selected = np.argmax(p2_selected, axis=1)
else:
    p2_selected = np.empty((0, len(CLASS_NAMES)), dtype=np.float32)
    y2_pred_selected = np.empty((0,), dtype=np.int32)
    model2_routed_seconds = 0.0

print("Sequences routed to Model 2:", len(model2_indices))
print(f"Model 2 routed inference: {model2_routed_seconds:.3f} s")

if len(model2_indices):
    print(
        "Average Model 2 batch inference per routed sequence: "
        f"{model2_routed_seconds / len(model2_indices) * 1000:.3f} ms"
    )

print("\nPredicted anomaly-type distribution:")
for class_id, class_name in enumerate(CLASS_NAMES):
    count = int(np.sum(y2_pred_selected == class_id))
    print(f"{class_id}: {class_name}: {count}")

In [ ]:
true_anomaly_indices = np.flatnonzero(y1_test == 1)

print("True anomaly positions in Model 1 test:", len(true_anomaly_indices))
print("Saved Model 2 anomaly labels:", len(y2_test))

assert len(true_anomaly_indices) == len(y2_test), (
    "The number of true anomaly rows in Model 1 does not match the "
    "number of Model 2 labels. Verify the saved split and sequence "
    "generation before evaluating end-to-end anomaly types."
)

assert np.all((y2_test >= 0) & (y2_test < len(CLASS_NAMES))), (
    "Unexpected Model 2 class IDs. Check the saved label mapping."
)

print("Counts match. Verify that both arrays preserve the same chronological "
      "order of anomaly target rows before interpreting type metrics.")

In [ ]:
# Model 2's predictions at the actual anomaly positions, for standalone
# multiclass evaluation using the complete-row feature sequences.
p2_true_anomalies = predict_in_batches(
    model2_xla_batch,
    X2_all_test[true_anomaly_indices],
    batch_size=256,
)
y2_pred_standalone = np.argmax(p2_true_anomalies, axis=1)

print("Model 2 standalone results on true anomaly sequences:")
print(classification_report(
    y2_test,
    y2_pred_standalone,
    labels=list(range(len(CLASS_NAMES))),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

print("Standalone confusion matrix:")
print(confusion_matrix(
    y2_test,
    y2_pred_standalone,
    labels=list(range(len(CLASS_NAMES))),
))

# ------------------------------------------------------------
# END-TO-END: Model 1 misses count as missed anomaly types.
# ------------------------------------------------------------
end_to_end_pred = np.full(len(y2_test), fill_value=len(CLASS_NAMES), dtype=int)

# Find true anomalies detected by Model 1.
detected_positions = y1_pred[true_anomaly_indices] == 1

# Model 2 prediction for each sequence routed to it.
routed_prediction_by_index = {
    int(test_index): int(class_prediction)
    for test_index, class_prediction in zip(model2_indices, y2_pred_selected)
}

for j, test_index in enumerate(true_anomaly_indices):
    if detected_positions[j]:
        end_to_end_pred[j] = routed_prediction_by_index[int(test_index)]

end_to_end_names = CLASS_NAMES + ["Missed by Model 1"]

print("\nEnd-to-end anomaly-type confusion matrix:")
print(confusion_matrix(
    y2_test,
    end_to_end_pred,
    labels=list(range(len(end_to_end_names))),
))

exact_type_correct = int(np.sum(end_to_end_pred == y2_test))
end_to_end_type_accuracy = exact_type_correct / len(y2_test)

print(f"\nEnd-to-end exact anomaly-type accuracy: "
      f"{end_to_end_type_accuracy:.4f}")
print(f"Correct type predictions: {exact_type_correct}/{len(y2_test)}")
print(f"True anomalies missed by Model 1: "
      f"{int(np.sum(~detected_positions))}/{len(y2_test)}")

if detected_positions.any():
    print("\nModel 2 type classification among true anomalies detected by Model 1:")
    print(classification_report(
        y2_test[detected_positions],
        end_to_end_pred[detected_positions],
        labels=list(range(len(CLASS_NAMES))),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
else:
    print("Model 1 detected no true anomalies; conditional type metrics unavailable.")

In [ ]:
def summarize_latency_ms(values):
    values = np.asarray(values, dtype=np.float64)

    return {
        "calls": len(values),
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p95_ms": float(np.percentile(values, 95)),
        "p99_ms": float(np.percentile(values, 99)),
        "max_ms": float(np.max(values)),
        "throughput_per_second": float(1000 / np.mean(values)),
    }


def print_latency_summary(title, values):
    stats = summarize_latency_ms(values)

    print(f"\n{title}")
    print("-" * len(title))
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"{key:>24}: {value:.3f}")
        else:
            print(f"{key:>24}: {value}")

    return stats

In [ ]:
# Use evenly spaced samples so the benchmark is not restricted to
# just the beginning of the test period.
BENCHMARK_SAMPLES = min(100, len(X1_test))

benchmark_indices = np.unique(
    np.linspace(
        0,
        len(X1_test) - 1,
        num=BENCHMARK_SAMPLES,
        dtype=int,
    )
)

# Warm-up ordinary Keras predict calls (not timed).
_ = model1.predict(X1_test[:1], verbose=0)
_ = model2.predict(X2_all_test[:1], verbose=0)

# Also warm up optimized single-sequence calls.
_ = model1_xla_one(tf.convert_to_tensor(X1_test[:1], dtype=tf.float32))
_ = model2_xla_one(tf.convert_to_tensor(X2_all_test[:1], dtype=tf.float32))

keras_model1_ms = []
keras_model2_ms = []
keras_pipeline_ms = []

xla_model1_ms = []
xla_model2_ms = []
xla_pipeline_ms = []

keras_predictions = []
xla_predictions = []

for idx in benchmark_indices:
    x1 = X1_test[idx:idx + 1]
    x2 = X2_all_test[idx:idx + 1]

    # ---------------- KERAS BASELINE ----------------
    t0 = time.perf_counter()
    out1 = model1.predict(x1, verbose=0)
    t1 = time.perf_counter()

    prob1 = float(np.asarray(out1).reshape(-1)[0])
    keras_model1_ms.append((t1 - t0) * 1000)

    type1 = None
    total_start = t0

    if prob1 >= MODEL1_THRESHOLD:
        t2 = time.perf_counter()
        out2 = model2.predict(x2, verbose=0)
        t3 = time.perf_counter()

        keras_model2_ms.append((t3 - t2) * 1000)
        type1 = int(np.argmax(out2[0]))

    keras_pipeline_ms.append((time.perf_counter() - total_start) * 1000)
    keras_predictions.append((prob1 >= MODEL1_THRESHOLD, type1))

    # ---------------- TENSORFLOW + XLA ----------------
    x1_tensor = tf.convert_to_tensor(x1, dtype=tf.float32)
    x2_tensor = tf.convert_to_tensor(x2, dtype=tf.float32)

    t0 = time.perf_counter()
    out1_xla = model1_xla_one(x1_tensor)
    # Materialize the result so timing includes execution.
    prob1_xla = float(out1_xla.numpy().reshape(-1)[0])
    t1 = time.perf_counter()

    xla_model1_ms.append((t1 - t0) * 1000)

    type2 = None

    if prob1_xla >= MODEL1_THRESHOLD:
        t2 = time.perf_counter()
        out2_xla = model2_xla_one(x2_tensor)
        type2 = int(np.argmax(out2_xla.numpy()[0]))
        t3 = time.perf_counter()

        xla_model2_ms.append((t3 - t2) * 1000)

    xla_pipeline_ms.append((time.perf_counter() - t0) * 1000)
    xla_predictions.append((prob1_xla >= MODEL1_THRESHOLD, type2))

print(f"Benchmark samples: {len(benchmark_indices)}")
print(f"Model 2 calls with Keras: {len(keras_model2_ms)}")
print(f"Model 2 calls with XLA:   {len(xla_model2_ms)}")

latency_results = {}

latency_results["Keras Model 1"] = print_latency_summary(
    "Keras Model 1 — one sequence", keras_model1_ms
)

if keras_model2_ms:
    latency_results["Keras Model 2"] = print_latency_summary(
        "Keras Model 2 — conditional calls", keras_model2_ms
    )

latency_results["Keras full pipeline"] = print_latency_summary(
    "Keras two-stage pipeline — one sequence", keras_pipeline_ms
)

latency_results["XLA Model 1"] = print_latency_summary(
    "TensorFlow/XLA Model 1 — one sequence", xla_model1_ms
)

if xla_model2_ms:
    latency_results["XLA Model 2"] = print_latency_summary(
        "TensorFlow/XLA Model 2 — conditional calls", xla_model2_ms
    )

latency_results["XLA full pipeline"] = print_latency_summary(
    "TensorFlow/XLA two-stage pipeline — one sequence", xla_pipeline_ms
)

In [ ]:
keras_predictions = np.asarray(keras_predictions, dtype=object)
xla_predictions = np.asarray(xla_predictions, dtype=object)

same_binary = np.mean(keras_predictions[:, 0] == xla_predictions[:, 0])

# Compare anomaly-type predictions only where both methods called Model 2.
both_routed = (
    (keras_predictions[:, 0] == True)
    & (xla_predictions[:, 0] == True)
)

if both_routed.any():
    same_type = np.mean(
        keras_predictions[both_routed, 1].astype(int)
        == xla_predictions[both_routed, 1].astype(int)
    )
else:
    same_type = np.nan

keras_mean = np.mean(keras_pipeline_ms)
xla_mean = np.mean(xla_pipeline_ms)
speedup = keras_mean / xla_mean if xla_mean > 0 else np.nan

print(f"Binary decision agreement: {same_binary:.2%}")
print(
    "Type prediction agreement when both routed to Model 2: "
    f"{same_type:.2%}" if not np.isnan(same_type)
    else "No samples were routed to Model 2 in the comparison."
)
print(f"Mean end-to-end Keras latency: {keras_mean:.3f} ms")
print(f"Mean end-to-end XLA latency:   {xla_mean:.3f} ms")
print(f"Observed end-to-end speedup:   {speedup:.2f}x")

In [ ]:
REPORT_DIR = PROJECT_ROOT / "reports" / "inference_baseline"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Model 1 sequences start at the 20th row of the complete test table.
test_target_rows = test_df.iloc[SEQUENCE_LENGTH - 1:].copy().reset_index(drop=True)

assert len(test_target_rows) == len(X1_test), (
    "Target rows do not align with Model 1 sequences. "
    "Check sequence-generation logic before exporting."
)

# Save the predicted class for rows routed to Model 2.
predicted_type_by_index = {
    int(idx): CLASS_NAMES[int(class_id)]
    for idx, class_id in zip(model2_indices, y2_pred_selected)
}

prediction_output = pd.DataFrame({
    "y_true_binary": y1_test.astype(int),
    "model1_anomaly_probability": p1.astype(float),
    "model1_predicted_anomaly": y1_pred.astype(int),
    "model2_predicted_type": [
        predicted_type_by_index.get(i, None)
        for i in range(len(X1_test))
    ],
})

# Preserve the timestamp if the table has one.
if "FECHA" in test_target_rows.columns:
    prediction_output.insert(0, "FECHA", test_target_rows["FECHA"].values)
elif "Fecha" in test_target_rows.columns:
    prediction_output.insert(0, "Fecha", test_target_rows["Fecha"].values)

predictions_path = REPORT_DIR / "end_to_end_test_predictions.csv"
prediction_output.to_csv(predictions_path, index=False)

latency_table = pd.DataFrame(latency_results).T
latency_path = REPORT_DIR / "latency_benchmark.csv"
latency_table.to_csv(latency_path, index=True)

print("Saved predictions:", predictions_path)
print("Saved latency metrics:", latency_path)
display(latency_table)